# refshift -- reference-mismatch decoder experiments

CSP+LDA, ShallowConvNet, EEGNet and ATCNet mismatch matrices (with and without
Euclidean Alignment), full jitter, leave-one-reference-out and
leave-one-family-out.

Set `DATASET` in section B and run top to bottom. Every runner caches its result
table to `RESULTS`, so a rerun reloads instead of retraining.

## A -- Setup (run once per kernel)

In [ ]:
import os, pathlib, subprocess, sys, importlib

os.chdir("/kaggle/working")
REFSHIFT_DIR = pathlib.Path("/kaggle/working/Reference-Mismatch-MI-Net")
if REFSHIFT_DIR.exists():
    subprocess.run(["rm", "-rf", str(REFSHIFT_DIR)], check=True)
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/JatinArutla/Reference-Mismatch-MI-Net",
                str(REFSHIFT_DIR)], check=True)

# sys.executable, not bare "pip": on Kaggle the two can be different interpreters.
PIP = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"]
# Pinned first, then the package with --no-deps so pip cannot quietly swap
# moabb or mne underneath us.
subprocess.run(PIP + ["moabb==1.5.0", "mne==1.11.0", "mne-bids>=0.18",
                      "braindecode>=1.0", "skorch>=0.15", "pyriemann>=0.5"], check=True)
subprocess.run(PIP + ["-e", str(REFSHIFT_DIR), "--no-deps"], check=True)

# `pip install -e` drops a .pth file into site-packages, and .pth files are read
# by `site` ONLY at interpreter startup. A kernel that is already running never
# sees it, so `import refshift` fails until you restart. Putting the repo root on
# sys.path directly gives the same result with no restart.
sys.path.insert(0, str(REFSHIFT_DIR))
importlib.invalidate_caches()

import refshift
print("refshift:", refshift.__file__)

In [ ]:
import warnings
import numpy as np
import pandas as pd
import mne

from refshift import (
    setup_kaggle_env, contrast_recovery_report, calibrate_csp_lda,
    run_mismatch, run_mismatch_jitter, run_loro_matrix, run_lofo_matrix,
    report_matrix, report_families, report_jitter_full, report_loro, report_lofo,
    transfer_gap_ci, reference_modes_for_dataset, canonical_mode_tuple,
)
from refshift.preprocess import load_windows
from refshift.datasets import subject_list

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

CACHE   = "/kaggle/working/cache"      # preprocessed windows (.npz)
RESULTS = "/kaggle/working/results"    # experiment tables (.csv)
for d in (CACHE, RESULTS):
    os.makedirs(d, exist_ok=True)

## B -- Config (the only cell you edit)

`setup_kaggle_env` symlinks the attached Kaggle datasets into MOABB's cache
layout. It raises if a dataset is not attached, rather than letting MOABB
download gigabytes over the network.

In [ ]:
DATASET    = "iv2a"     # iv2a | openbmi | schirrmeister2017 | dreyer2023
REFERENCES = None       # None -> the dataset-safe default set
SEEDS      = [0, 1, 2]  # deep nets average these; CSP+LDA is deterministic
MAX_EPOCHS, BATCH_SIZE = 200, 32

setup_kaggle_env(symlink_datasets=[DATASET])

MODES = (reference_modes_for_dataset(DATASET) if REFERENCES is None
         else canonical_mode_tuple(REFERENCES))
SUBJECTS = subject_list(DATASET)
print(f"{DATASET}: {len(SUBJECTS)} subjects, {len(MODES)} references {MODES}")

## C -- Calibration

Bare CSP+LDA through MOABB's own evaluation loop, against its published
65.99% on IV-2a. Independent of this package's loaders.

In [ ]:
if DATASET == "iv2a":
    _, score, passed = calibrate_csp_lda("iv2a")
    print("calibration passed:", passed)
else:
    print("no published target for", DATASET)

## D -- Operator invertibility (algebraic, data-free)

Per operator: is it canonicalizable by re-referencing (`contrast_preserving`,
H.M = H), are the native channel contrasts still linearly recoverable from its
output, and how well conditioned is that recovery. Global references collapse
to a common reference; the Laplacians change the coordinate system but lose no
contrast information, so a fixed decoder's failure on them is a coordinate
mismatch rather than lost information.

In [ ]:
_, _, _, _, CH_NAMES = load_windows(DATASET, SUBJECTS[0], cache_dir=CACHE)
inv = contrast_recovery_report(CH_NAMES, modes=MODES)
print(inv.to_string(index=False))
inv.to_csv(f"{RESULTS}/{DATASET}_operator_invertibility.csv", index=False)

## E -- Mismatch matrices, no EA

In [ ]:
df_csp = run_mismatch(DATASET, model="csp_lda", seeds=[0],
                      reference_modes=REFERENCES, apply_ea=False,
                      cache_dir=CACHE, results_dir=RESULTS)
report_matrix(df_csp, title=f"CSP+LDA (no EA) -- {DATASET}", modes=MODES)
report_families(df_csp, title=f"CSP+LDA (no EA) -- {DATASET}", modes=MODES)

In [ ]:
df_shallow = run_mismatch(DATASET, model="shallow", seeds=SEEDS,
                          reference_modes=REFERENCES, apply_ea=False,
                          max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                          cache_dir=CACHE, results_dir=RESULTS)
report_matrix(df_shallow, title=f"ShallowConvNet (no EA) -- {DATASET}", modes=MODES)
report_families(df_shallow, title=f"ShallowConvNet (no EA) -- {DATASET}", modes=MODES)

In [ ]:
df_eegnet = run_mismatch(DATASET, model="eegnet", seeds=SEEDS,
                         reference_modes=REFERENCES, apply_ea=False,
                         max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                         cache_dir=CACHE, results_dir=RESULTS)
report_matrix(df_eegnet, title=f"EEGNet (no EA) -- {DATASET}", modes=MODES)

In [ ]:
df_atcnet = run_mismatch(DATASET, model="atcnet", seeds=SEEDS,
                         reference_modes=REFERENCES, apply_ea=False,
                         max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                         cache_dir=CACHE, results_dir=RESULTS)
report_matrix(df_atcnet, title=f"ATCNet (no EA) -- {DATASET}", modes=MODES)

## F -- Mismatch matrices, with EA

EA is fit per block, train and test independently, after referencing. If it
collapses the gap, the shift is second-order covariance geometry rather than
lost task information.

In [ ]:
df_csp_ea = run_mismatch(DATASET, model="csp_lda", seeds=[0],
                         reference_modes=REFERENCES, apply_ea=True,
                         cache_dir=CACHE, results_dir=RESULTS)
report_matrix(df_csp_ea, title=f"CSP+LDA (EA) -- {DATASET}", modes=MODES)

In [ ]:
df_shallow_ea = run_mismatch(DATASET, model="shallow", seeds=SEEDS,
                             reference_modes=REFERENCES, apply_ea=True,
                             max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                             cache_dir=CACHE, results_dir=RESULTS)
report_matrix(df_shallow_ea, title=f"ShallowConvNet (EA) -- {DATASET}", modes=MODES)

## G -- Full jitter

Train one net with a different reference drawn per sample, then test on every
reference. A small spread means the model learned to be reference-invariant.

In [ ]:
df_jitter = run_mismatch_jitter(DATASET, model="shallow", condition="full",
                                seeds=SEEDS, reference_modes=REFERENCES,
                                max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                                cache_dir=CACHE, results_dir=RESULTS)
report_jitter_full(df_jitter, title=f"Full jitter -- ShallowConvNet -- {DATASET}",
                   modes=MODES)

## H -- Leave-one-reference-out

Hold one reference out of the jitter mix, train on the rest, test on all of
them. The diagonal is the unseen reference; the recovery gap is the cost of
never training on it. Each holdout caches separately, so an interrupted sweep
resumes.

In [ ]:
df_loro = run_loro_matrix(DATASET, model="shallow", seeds=SEEDS,
                          reference_modes=REFERENCES,
                          max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                          cache_dir=CACHE, results_dir=RESULTS)
report_loro(df_loro, title=f"LORO -- ShallowConvNet -- {DATASET}", modes=MODES)

## I -- Leave-one-family-out

Hold out a whole family (global / single / spatial) and test whether the model
generalises across *kinds* of reference operation.

In [ ]:
df_lofo = run_lofo_matrix(DATASET, model="shallow", seeds=SEEDS,
                          reference_modes=REFERENCES,
                          max_epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                          cache_dir=CACHE, results_dir=RESULTS)
report_lofo(df_lofo, title=f"LOFO -- ShallowConvNet -- {DATASET}")

## J -- Summary

Subject-level transfer gap with a bootstrap CI for each run. This is the table
that goes in the paper.

In [ ]:
runs = [
    ("CSP+LDA",        "no EA", df_csp),
    ("ShallowConvNet", "no EA", df_shallow),
    ("EEGNet",         "no EA", df_eegnet),
    ("ATCNet",         "no EA", df_atcnet),
    ("CSP+LDA",        "EA",    df_csp_ea),
    ("ShallowConvNet", "EA",    df_shallow_ea),
]
summary = pd.DataFrame([
    {"model": m, "condition": c, **transfer_gap_ci(df)} for m, c, df in runs
])
print(summary.to_string(index=False))
summary.to_csv(f"{RESULTS}/{DATASET}_transfer_gaps.csv", index=False)